### Transformar Datos de "Locations"
1. Eliminar registros NULL del campo "location_id"
2. Eliminar registros duplicados "identicos"
3. Eliminar registros duplicados en base al campo "created_timestamp"
4. Convertir las columnas al "tipo de dato" correcto
5. Escribir los datos "transformados" en el schema "silver"

In [0]:
locations_df = spark.sql("SELECT * FROM zenviro.bronze.locations_py")
display(locations_df)

In [0]:
locations_df = spark.table("zenviro.bronze.locations_py")
display(locations_df)

In [0]:
locations_df = spark.read.table("zenviro.bronze.locations_py")
display(locations_df)

#### 1. Eliminar registros NULL del campo "location_id"

In [0]:
#Instrucción SQL
locations_filtered_df = locations_df.filter("location_id is not null")
display(locations_filtered_df)

In [0]:
#Notación de Objetos
locations_filtered_df = locations_df.filter(locations_df.location_id.isNotNull())
display(locations_filtered_df)

In [0]:
#"Filter" y ""Where son sinónimos
locations_filtered_df = locations_df.where(locations_df.location_id.isNotNull())
display(locations_filtered_df)

#### 2. Eliminar registros duplicados "identicos"

In [0]:
#Ordenar los datos por la columnas "location_id"
locations_filtered_df = locations_df.where(locations_df.location_id.isNotNull()).orderBy("location_id")
display(locations_filtered_df)

In [0]:
#Función "distinct"
locations_distinct_df = locations_filtered_df.distinct().orderBy("location_id")
display(locations_distinct_df)

In [0]:
#Función "dropDuplicates"
locations_distinct_df = locations_filtered_df.dropDuplicates().orderBy("location_id")
display(locations_distinct_df)

In [0]:
%skip
%sql
-- Primera Forma
SELECT location_id,
       MAX(created_timestamp),
       MAX(location_name),
       MAX(address),
       MAX(city),
       MAX(country)
FROM zenviro.bronze.v_locations
WHERE location_id IS NOT NULL
GROUP BY location_id
ORDER BY location_id;

In [0]:
from pyspark.sql.functions import max

max_ts_df = locations_distinct_df.groupBy("location_id") \
            .agg(max("created_timestamp").alias("max_created_timestamp")) \
            .orderBy("location_id")
display(max_ts_df)

#### 3. Eliminar registros duplicados en base al campo "created_timestamp"

In [0]:
locations_distinct_ct_df = (
    locations_distinct_df
    .join(max_ts_df,
          (locations_distinct_df.location_id == max_ts_df.location_id) &
          (locations_distinct_df.created_timestamp == max_ts_df.max_created_timestamp),
          "inner")
    .select(locations_distinct_df["*"])
)
display(locations_distinct_ct_df)

#### 4. Convertir las columnas al "tipo de dato" correcto

In [0]:
locations_casted_df = (
    locations_distinct_ct_df
    .select(locations_distinct_ct_df.created_timestamp.cast("timestamp"),
            locations_distinct_ct_df.location_id,
            locations_distinct_ct_df.location_name,
            locations_distinct_ct_df.address,
            locations_distinct_ct_df.city,
            locations_distinct_ct_df.country)
)
display(locations_casted_df)

#### 5. Escribir los datos "transformados" en el schema "silver"

In [0]:
locations_casted_df.writeTo("zenviro.silver.locations_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.locations_py"))